# 08 — Model Selection & Hyperparameter Tuning

**Topics:** GridSearchCV, RandomizedSearchCV, learning curves, validation curves, bias-variance tradeoff, nested CV.

**Reference:** [sklearn model selection](https://scikit-learn.org/stable/model_selection.html)

**Dataset:** Wine Quality — multi-feature regression/classification on a real sensory dataset.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, KFold, cross_val_score, cross_validate,
    learning_curve, validation_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score, make_scorer
import warnings
warnings.filterwarnings('ignore')

wine_raw = fetch_openml(name='wine-quality-red', version=1, as_frame=True, parser='auto').frame
wine = wine_raw.copy()
wine.columns = [c.strip().replace(' ', '_') for c in wine.columns]
# Binary target: quality >= 6 is 'good'
wine['target'] = (wine['class'].astype(float) >= 6).astype(int)
X = wine.drop(['class', 'target'], axis=1).astype(float)
y = wine['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Dataset: {X.shape} | Class balance: {y.value_counts().to_dict()}")

---
## Exercise 1 — GridSearchCV

**Task:** Tune a `RandomForestClassifier` with an exhaustive grid search.

1. Define a pipeline: `StandardScaler` → `RandomForestClassifier(random_state=42)`.
2. Define a parameter grid covering:
   - `n_estimators`: [50, 100, 200]
   - `max_depth`: [3, 5, None]
   - `min_samples_split`: [2, 5]
3. Run `GridSearchCV` with 5-fold stratified CV, scoring='f1', `n_jobs=-1`.
4. Return `grid_results`: DataFrame of all parameter combinations + `mean_test_score`, `std_test_score`, `rank_test_score`. Sorted by rank.
5. Return `best_params` and `best_cv_score`.

In [ ]:
def run_grid_search(X_train, y_train):
    """
    Returns (grid_search_fitted, grid_results, best_params, best_cv_score)
    """
    # YOUR CODE HERE
    pass

gs, grid_results, best_params, best_cv_score = run_grid_search(X_train, y_train)

In [ ]:
# --- ASSERTIONS ---
assert len(grid_results) == 3 * 3 * 2, f"Expected 18 combinations, got {len(grid_results)}"
assert 'mean_test_score' in grid_results.columns
assert 'rank_test_score' in grid_results.columns
assert grid_results['rank_test_score'].iloc[0] == 1, "First row must be rank 1"
assert isinstance(best_params, dict)
assert 0.5 < best_cv_score <= 1.0
print(f"✓ Exercise 1 passed — Best CV F1: {best_cv_score:.4f}")
print(f"Best params: {best_params}")

---
## Exercise 2 — RandomizedSearchCV

**Task:** Compare RandomizedSearchCV vs GridSearchCV on efficiency and result quality.

1. Define a **wider** parameter distribution for `RandomForestClassifier`:
   - `n_estimators`: uniform int 50–500
   - `max_depth`: [2, 3, 5, 8, 10, None]
   - `min_samples_split`: [2, 5, 10, 20]
   - `max_features`: ['sqrt', 'log2', 0.5]
2. Run `RandomizedSearchCV` with `n_iter=30`, 5-fold CV, scoring='f1', `random_state=42`.
3. Compare to `GridSearchCV` result: which found a better F1? How many fits did each require?
4. Return `comparison`: dict with keys `grid_best_f1`, `random_best_f1`, `grid_n_fits`, `random_n_fits`.

In [ ]:
from scipy.stats import randint

def run_randomized_search(X_train, y_train, grid_best_score):
    """
    Returns (rs_fitted, comparison dict)
    """
    # YOUR CODE HERE
    pass

rs, comparison = run_randomized_search(X_train, y_train, best_cv_score)

In [ ]:
# --- ASSERTIONS ---
assert set(comparison.keys()) == {'grid_best_f1', 'random_best_f1', 'grid_n_fits', 'random_n_fits'}
assert comparison['grid_n_fits'] == 18 * 5, "Grid should run 18*5=90 fits"
assert comparison['random_n_fits'] == 30 * 5, "Random should run 30*5=150 fits"
assert comparison['random_best_f1'] > 0.5
print(f"✓ Exercise 2 passed")
print(f"Grid: {comparison['grid_best_f1']:.4f} ({comparison['grid_n_fits']} fits) | "
      f"Random: {comparison['random_best_f1']:.4f} ({comparison['random_n_fits']} fits)")

---
## Exercise 3 — Validation Curve (Bias-Variance)

**Task:** Use `validation_curve` to diagnose bias vs variance for a single hyperparameter.

1. Compute validation curve for `RandomForestClassifier` over `n_estimators` = [10, 25, 50, 75, 100, 150, 200, 300]. Param name: `randomforestclassifier__n_estimators` (inside pipeline).
2. Record mean and std of train and CV scores.
3. Return `val_curve_df`: columns `n_estimators`, `train_mean`, `train_std`, `cv_mean`, `cv_std`.
4. In a markdown cell: identify the region of underfitting and the region where more estimators stop helping.

In [ ]:
def compute_validation_curve(X_train, y_train):
    """
    Returns val_curve_df DataFrame.
    """
    # YOUR CODE HERE
    pass

val_curve_df = compute_validation_curve(X_train, y_train)

In [ ]:
# --- ASSERTIONS ---
assert list(val_curve_df.columns) == ['n_estimators', 'train_mean', 'train_std', 'cv_mean', 'cv_std']
assert len(val_curve_df) == 8
assert (val_curve_df['train_mean'] >= val_curve_df['cv_mean']).all(), "Train score should be >= CV score"
print("✓ Exercise 3 passed")
print(val_curve_df)

**Observation:** *(Write here — where does underfitting occur? Where does adding estimators stop helping?)*

---
## Exercise 4 — Learning Curve (Training Size Effect)

**Task:** Use sklearn's `learning_curve` to understand how much data the model needs.

1. Compute learning curve for the best RF from Exercise 1 over train sizes `[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]`.
2. Return `lc_df`: columns `train_size_n`, `train_mean`, `train_std`, `cv_mean`, `cv_std`.
3. Compute `convergence_point`: the smallest `train_size_n` where `cv_mean` is within 2% of its maximum value.
4. Answer in markdown: is this model data-hungry or does it converge quickly?

In [ ]:
def compute_learning_curve(model, X_train, y_train):
    """
    Returns (lc_df, convergence_point)
    """
    # YOUR CODE HERE
    pass

lc_df, convergence_point = compute_learning_curve(gs.best_estimator_, X_train, y_train)

In [ ]:
# --- ASSERTIONS ---
assert list(lc_df.columns) == ['train_size_n', 'train_mean', 'train_std', 'cv_mean', 'cv_std']
assert len(lc_df) == 9
assert isinstance(convergence_point, (int, np.integer))
assert convergence_point <= lc_df['train_size_n'].max()
print(f"✓ Exercise 4 passed — Convergence at ~{convergence_point} samples")
print(lc_df)

---
## Exercise 5 — Nested Cross-Validation

**Concept:** Standard CV reuses the test fold for model selection → optimistically biased. Nested CV gives an unbiased estimate of the true generalization error.

Implement **nested CV**:
- Outer loop: 5-fold stratified CV (estimates generalization)
- Inner loop: 3-fold CV inside GridSearchCV (selects hyperparameters)

1. For each outer fold, run GridSearchCV on the training portion, evaluate best model on held-out fold.
2. Return `nested_scores`: array of 5 F1 scores (one per outer fold).
3. Return `nested_mean`, `nested_std`.
4. Compare to the non-nested CV score from Exercise 1. What's the optimism bias?

In [ ]:
def nested_cross_validation(X, y):
    """
    Returns (nested_scores array, nested_mean, nested_std)
    """
    # YOUR CODE HERE
    pass

nested_scores, nested_mean, nested_std = nested_cross_validation(X, y)

In [ ]:
# --- ASSERTIONS ---
assert len(nested_scores) == 5
assert (nested_scores > 0.4).all(), "All outer folds should achieve F1 > 0.4"
optimism_bias = best_cv_score - nested_mean
print(f"✓ Exercise 5 passed")
print(f"Nested CV: {nested_mean:.4f} ± {nested_std:.4f}")
print(f"Non-nested CV: {best_cv_score:.4f}")
print(f"Optimism bias: {optimism_bias:.4f}")

---
## Exercise 6 — Pipeline Search with ColumnTransformer

**Task:** Run a grid search over a full pipeline including preprocessing steps — the real-world pattern.

Build a pipeline:
- `scaler`: StandardScaler (to be toggled in the search)
- `clf`: LogisticRegression

Grid to search:
- Whether to scale or not: `scaler` = [StandardScaler(), 'passthrough']
- `clf__C`: [0.001, 0.01, 0.1, 1, 10]
- `clf__penalty`: ['l1', 'l2'] (use solver='saga')

1. Run GridSearchCV with 5-fold CV, scoring='roc_auc'.
2. Return `pipe_grid_results`: top 10 combinations by mean test score.
3. Return `best_pipe`: the best fitted pipeline.

In [ ]:
def search_full_pipeline(X_train, y_train):
    """
    Returns (best_pipe, pipe_grid_results)
    """
    # YOUR CODE HERE
    pass

best_pipe, pipe_grid_results = search_full_pipeline(X_train, y_train)

In [ ]:
# --- ASSERTIONS ---
assert len(pipe_grid_results) == 10
assert 'mean_test_score' in pipe_grid_results.columns
assert isinstance(best_pipe, Pipeline)
test_auc = roc_auc_score(y_test, best_pipe.predict_proba(X_test)[:, 1]) if hasattr(best_pipe.named_steps['clf'], 'predict_proba') else 0
from sklearn.metrics import roc_auc_score
print(f"✓ Exercise 6 passed")
print(pipe_grid_results[['param_scaler', 'param_clf__C', 'param_clf__penalty', 'mean_test_score']].head())

---
## Exercise 7 — Model Selection Report

**Task:** Produce a final model selection report — the kind you'd present to a technical panel.

Using all models tuned in this notebook, build `selection_report`: a DataFrame comparing:
- Best RF (GridSearchCV)
- Best RF (RandomizedSearchCV)
- Best LogisticRegression (pipeline search)

Columns: `model`, `best_params_summary` (string), `cv_score`, `test_f1`, `test_roc_auc`, `n_fits`.

Sort by `test_roc_auc` descending. Include a markdown cell with your recommendation and reasoning.

In [ ]:
from sklearn.metrics import roc_auc_score

def build_selection_report(gs, rs, best_pipe, X_test, y_test):
    """
    Returns selection_report DataFrame.
    """
    # YOUR CODE HERE
    pass

selection_report = build_selection_report(gs, rs, best_pipe, X_test, y_test)

In [ ]:
# --- ASSERTIONS ---
assert len(selection_report) == 3
for col in ['model', 'best_params_summary', 'cv_score', 'test_f1', 'test_roc_auc', 'n_fits']:
    assert col in selection_report.columns, f"Missing: {col}"
assert selection_report['test_roc_auc'].is_monotonic_decreasing
print("✓ Exercise 7 passed")
print(selection_report)

**Recommendation:** *(Write your model choice and reasoning here — consider: performance, interpretability, training cost, overfitting risk)*